# Coding Assignment 2: Advanced Ensemble Learning and Evaluation for Cancer Prediction
# ID 2671507 | Abdul Khaled Arafat

## Section A: Data Engineering & Feature Selection

In [ ]:
import pandas as pd
from sklearn.datasets import load_breast_cancer

# Load the Breast Cancer Wisconsin dataset
bc_data = load_breast_cancer()

# Convert to a Pandas DataFrame
df = pd.DataFrame(data=bc_data.data, columns=bc_data.feature_names)
df['target'] = bc_data.target

display(df.head())
print(df.info())

### Data Integrity Check: Missing Values

In [ ]:
# Check for missing values
missing_values = df.isnull().sum()
missing_values = missing_values[missing_values > 0]

if not missing_values.empty:
    print("Found missing values in the following columns:")
    print(missing_values)
else:
    print("No missing values found in the dataset.")

### Feature Engineering: Top 5 Correlated Features

In [ ]:
# Calculate correlations with the target variable
correlations = df.corr()['target'].abs().sort_values(ascending=False)

# Select the top 5 features (excluding the target if it appears in the top 5)
top_5_features = correlations[1:6].index.tolist()

print(f"Top 5 features correlated with the target variable: {top_5_features}")

# Create a new DataFrame with selected features and the target
X = df[top_5_features]
y = df['target']

### Feature Scaling: StandardScaler

In [ ]:
from sklearn.preprocessing import StandardScaler

# Initialize the StandardScaler
scaler = StandardScaler()

# Apply StandardScaler to the feature subset X
X_scaled = scaler.fit_transform(X)

X_scaled_df = pd.DataFrame(X_scaled, columns=X.columns)

print("Scaled features (first 5 rows):")
display(X_scaled_df.head())
print(f"Mean of scaled features (should be close to 0): {X_scaled_df.mean().mean():.2f}")
print(f"Standard deviation of scaled features (should be close to 1): {X_scaled_df.std().mean():.2f}")

## Section B: Model Implementation & Tuning

### 1. Decision Tree Classifier (Baseline Model)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, confusion_matrix

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.3, random_state=42)

# Initialize and train the Decision Tree Classifier
dt_classifier = DecisionTreeClassifier(random_state=42)
dt_classifier.fit(X_train, y_train)

# Make predictions
y_pred_dt = dt_classifier.predict(X_test)
y_proba_dt = dt_classifier.predict_proba(X_test)[:, 1]

# Evaluate the model
accuracy_dt = accuracy_score(y_test, y_pred_dt)
f1_dt = f1_score(y_test, y_pred_dt)
roc_auc_dt = roc_auc_score(y_test, y_proba_dt)

print(f"Decision Tree Classifier Accuracy: {accuracy_dt:.4f}")
print(f"Decision Tree Classifier F1-Score: {f1_dt:.4f}")
print(f"Decision Tree Classifier ROC-AUC Score: {roc_auc_dt:.4f}")

### 2. Gradient Boosting Classifier (Advanced Boosting Ensemble)

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import GridSearchCV

# Initialize Gradient Boosting Classifier
gb_classifier = GradientBoostingClassifier(random_state=42)

# Define hyperparameter grid for GridSearchCV
param_grid_gb = {
    'n_estimators': [50, 100, 200],
    'learning_rate': [0.05, 0.1, 0.2],
    'max_depth': [3, 5, 7]
}

# Set up GridSearchCV
grid_search_gb = GridSearchCV(estimator=gb_classifier, param_grid=param_grid_gb,
                              cv=5, scoring='roc_auc', n_jobs=-1, verbose=1)

# Fit GridSearchCV to the training data
grid_search_gb.fit(X_train, y_train)

# Get the best estimator
best_gb_classifier = grid_search_gb.best_estimator_

print(f"Best Hyperparameters for Gradient Boosting: {grid_search_gb.best_params_}")

# Make predictions with the best estimator
y_pred_gb = best_gb_classifier.predict(X_test)
y_proba_gb = best_gb_classifier.predict_proba(X_test)[:, 1]

# Evaluate the best model
accuracy_gb = accuracy_score(y_test, y_pred_gb)
f1_gb = f1_score(y_test, y_pred_gb)
roc_auc_gb = roc_auc_score(y_test, y_proba_gb)

print(f"Gradient Boosting Classifier Accuracy: {accuracy_gb:.4f}")
print(f"Gradient Boosting Classifier F1-Score: {f1_gb:.4f}")
print(f"Gradient Boosting Classifier ROC-AUC Score: {roc_auc_gb:.4f}")

### 3. Support Vector Machine (SVM) with a non-linear kernel and Hyperparameter Tuning

In [ ]:
from sklearn.svm import SVC

# Initialize SVM Classifier
# probability=True for ROC-AUC scoring
svm_classifier = SVC(random_state=42, probability=True)

# Define hyperparameter grid for GridSearchCV for SVM
param_grid_svm = {
    'C': [0.1, 1, 10, 100],
    'gamma': [0.001, 0.01, 0.1, 1],
    'kernel': ['rbf']
}

# Set up GridSearchCV
grid_search_svm = GridSearchCV(estimator=svm_classifier, param_grid=param_grid_svm,
                             cv=5, scoring='roc_auc', n_jobs=-1, verbose=1)

# Fit GridSearchCV to the training data
grid_search_svm.fit(X_train, y_train)

# Get the best estimator
best_svm_classifier = grid_search_svm.best_estimator_

print(f"Best Hyperparameters for SVM: {grid_search_svm.best_params_}")

# Make predictions with the best estimator
y_pred_svm = best_svm_classifier.predict(X_test)
y_proba_svm = best_svm_classifier.predict_proba(X_test)[:, 1]

# Evaluate the best model
accuracy_svm = accuracy_score(y_test, y_pred_svm)
f1_svm = f1_score(y_test, y_pred_svm)
roc_auc_svm = roc_auc_score(y_test, y_proba_svm)

print(f"SVM Classifier Accuracy: {accuracy_svm:.4f}")
print(f"SVM Classifier F1-Score: {f1_svm:.4f}")
print(f"SVM Classifier ROC-AUC Score: {roc_auc_svm:.4f}")

## Section C: Visualization & Advanced Evaluation

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from matplotlib.patches import Rectangle
from sklearn.model_selection import cross_validate
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import confusion_matrix

# Create a folder for all saved plots
output_folder = "phase_c_outputs"
os.makedirs(output_folder, exist_ok=True)

### Hyperparameter Impact Plot (Decision Tree - max_depth)



In [ ]:
# Values of max_depth to evaluate
depth_values = range(1, 16)

mean_training_accuracy = []
mean_validation_accuracy = []

# Evaluate each maximum depth using 5-fold cross-validation
for depth in depth_values:
    dt_model = DecisionTreeClassifier(
        max_depth=depth,
        random_state=42
    )

    cv_results = cross_validate(
        estimator=dt_model,
        X=X_train,
        y=y_train,
        cv=5,
        scoring="accuracy",
        return_train_score=True
    )

    mean_training_accuracy.append(
        cv_results["train_score"].mean()
    )

    mean_validation_accuracy.append(
        cv_results["test_score"].mean()
    )

# Create the line plot
plt.figure(figsize=(10, 6))

plt.plot(
    depth_values,
    mean_training_accuracy,
    marker="o",
    label="Training Accuracy"
)

plt.plot(
    depth_values,
    mean_validation_accuracy,
    marker="s",
    label="Validation Accuracy"
)

plt.title("Impact of Decision Tree Maximum Depth on Accuracy")
plt.xlabel("Maximum Tree Depth")
plt.ylabel("Mean Accuracy")
plt.xticks(list(depth_values))
plt.ylim(0, 1.05)
plt.grid(True, alpha=0.3)
plt.legend()
plt.tight_layout()

# Save the plot
hyperparameter_plot_path = os.path.join(
    output_folder,
    "hyperparameter_impact_plot.png"
)

plt.savefig(
    hyperparameter_plot_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print(f"Hyperparameter plot saved as: {hyperparameter_plot_path}")

### Model Comparison Matrix

In [ ]:
# Collect the evaluation metrics from section B
metrics_df = pd.DataFrame({
    "Model": [
        "Decision Tree",
        "Gradient Boosting",
        "SVM"
    ],
    "Accuracy": [
        accuracy_dt,
        accuracy_gb,
        accuracy_svm
    ],
    "F1-Score": [
        f1_dt,
        f1_gb,
        f1_svm
    ],
    "ROC-AUC Score": [
        roc_auc_dt,
        roc_auc_gb,
        roc_auc_svm
    ]
})

# Display the numerical results
display(metrics_df.round(4))

# Convert the DataFrame into long format for Seaborn
metrics_melted = metrics_df.melt(
    id_vars="Model",
    var_name="Metric",
    value_name="Score"
)

# Create the grouped bar chart
plt.figure(figsize=(12, 7))

ax = sns.barplot(
    data=metrics_melted,
    x="Model",
    y="Score",
    hue="Metric",
    palette="viridis"
)

plt.title("Model Comparison: Accuracy, F1-Score, and ROC-AUC Score")
plt.xlabel("Classification Model")
plt.ylabel("Score")
plt.ylim(0, 1.05)
plt.grid(
    axis="y",
    linestyle="--",
    alpha=0.5
)

plt.legend(
    title="Evaluation Metric",
    bbox_to_anchor=(1.02, 1),
    loc="upper left"
)

# Display values above the bars
for container in ax.containers:
    ax.bar_label(
        container,
        fmt="%.3f",
        padding=3,
        fontsize=9
    )

plt.tight_layout()

# Save the plot
comparison_plot_path = os.path.join(
    output_folder,
    "model_comparison_matrix.png"
)

plt.savefig(
    comparison_plot_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print(f"Model comparison plot saved as: {comparison_plot_path}")

### Confusion Matrix Heatmap

In [ ]:
# Class definitions in the Breast Cancer Wisconsin dataset:
# 0 = Malignant
# 1 = Benign

# Reorder the labels so that:
# Negative class = Benign (1)
# Positive class = Malignant (0)

# This produces the standard confusion matrix arrangement:
# [[TN, FP],
#  [FN, TP]]

cm_gb = confusion_matrix(
    y_test,
    y_pred_gb,
    labels=[1, 0]
)

# Extract the four confusion matrix values
tn, fp, fn, tp = cm_gb.ravel()

print("Gradient Boosting Confusion Matrix Results")
print("--------------------------------------------")
print(f"True Negatives  (Benign predicted as Benign):       {tn}")
print(f"False Positives (Benign predicted as Malignant):    {fp}")
print(f"False Negatives (Malignant predicted as Benign):    {fn}")
print(f"True Positives  (Malignant predicted as Malignant): {tp}")

# Create detailed labels for each cell
confusion_labels = np.array([
    [
        f"True Negative\n{tn}",
        f"False Positive\n{fp}"
    ],
    [
        f"False Negative\n{fn}",
        f"True Positive\n{tp}"
    ]
])

plt.figure(figsize=(9, 7))

ax = sns.heatmap(
    cm_gb,
    annot=confusion_labels,
    fmt="",
    cmap="Blues",
    cbar=False,
    linewidths=1,
    linecolor="white",
    xticklabels=[
        "Benign\n(Negative)",
        "Malignant\n(Positive)"
    ],
    yticklabels=[
        "Benign\n(Negative)",
        "Malignant\n(Positive)"
    ],
    annot_kws={
        "fontsize": 12
    }
)

plt.title(
    "Confusion Matrix: Optimized Gradient Boosting Classifier",
    pad=15
)

plt.xlabel("Predicted Diagnosis")
plt.ylabel("Actual Diagnosis")

# Highlight the False Positive cell
ax.add_patch(
    Rectangle(
        (1, 0),
        1,
        1,
        fill=False,
        edgecolor="red",
        linewidth=3
    )
)

# Highlight the False Negative cell
ax.add_patch(
    Rectangle(
        (0, 1),
        1,
        1,
        fill=False,
        edgecolor="red",
        linewidth=3
    )
)

plt.tight_layout()

# Save the plot
confusion_matrix_path = os.path.join(
    output_folder,
    "gradient_boosting_confusion_matrix.png"
)

plt.savefig(
    confusion_matrix_path,
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print(f"Confusion matrix saved as: {confusion_matrix_path}")

In [ ]:
print("Saved Phase C files:")
print(f"1. {hyperparameter_plot_path}")
print(f"2. {comparison_plot_path}")
print(f"3. {confusion_matrix_path}")